# 06 - Kaggle GPU Setup

Preparation and inference checks only. Attach the prepared setup input (the original ZIP or its automatically extracted folder) and the supplied competition dataset. Select a GPU accelerator and run all cells. No package installation, optimisation, backward pass, test-image loading or training is performed. CUDA execution has **not** been verified locally.

The ZIP hash and extracted transfer-manifest hash below pin the reviewed bundle. If you intentionally rebuild it, review the changes and update that pin to the SHA-256 printed by `scripts/prepare_kaggle_bundle.py`. Do not relax checksums or preprocessing fingerprints to bypass a failure. A different hosted dependency version is acceptable only if the strict fingerprints still match. Restart the session if another `src` package is already loaded.

## 1. Explicit paths, pinned bundle and failure report

Prefer the automatically extracted setup at `/kaggle/input/datasets/leocaoprojects/enge707-gpu-setup/`. Verify its pinned transfer-manifest SHA-256 before trusting its payload checksums. If that directory is absent, retain the original route requiring exactly one matching ZIP under `/kaggle/input` and its full pinned hash. Failures are saved to `/kaggle/working/gpu_setup_check.json`.

In [ ]:
from pathlib import Path, PurePosixPath, PureWindowsPath
from collections import Counter
from importlib.metadata import version
import hashlib
import importlib
import json
import platform
import stat
import struct
import sys
import zipfile

INPUT_ROOT=Path("/kaggle/input")
DATA_ROOT=Path("/kaggle/input/datasets/leocaoprojects/enge707-2026-s2/")
WORKING_ROOT=Path("/kaggle/working")
SETUP_ROOT=WORKING_ROOT/"enge707_gpu_setup"
REPORT_PATH=WORKING_ROOT/"gpu_setup_check.json"
BUNDLE_NAME="enge707_gpu_setup.zip"
EXTRACTED_BUNDLE_ROOT=Path("/kaggle/input/datasets/leocaoprojects/enge707-gpu-setup/")
EXPECTED_TRANSFER_MANIFEST_SHA256="14b63f88302e255b3f2acca0a34377ee35e5dc5fddada8136651d236d36fc8b8"
EXPECTED_BUNDLE_SHA256="e858d40c2aa73096d5eab2fdf8d376f362f33e118924dc7be0269b353c59574f"
EXPECTED_MEMBERS={"src/__init__.py","src/data_pipeline.py","src/models.py",
                  "data/processed/split_manifest.csv","data/processed/class_mapping.json",
                  "preprocessing_fingerprints.json","transfer_manifest.json"}
report={"status":"pending","stage":"initialisation","versions":{"python":platform.python_version()},
        "checks":{},"failures":[],"gpu_check_executed":False}

def save_report():
    WORKING_ROOT.mkdir(parents=True,exist_ok=True)
    REPORT_PATH.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")

def require(name,condition,detail=None):
    report["checks"][name]=bool(condition)
    if not condition:
        raise ValueError(detail or name)
    print(name+": PASS")

def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream,"sha256").hexdigest()

save_report()


In [ ]:
try:
    report["stage"]="bundle discovery"
    require("competition_data_root_exists",DATA_ROOT.is_dir())
    if EXTRACTED_BUNDLE_ROOT.exists() or EXTRACTED_BUNDLE_ROOT.is_symlink():
        require("extracted_bundle_is_directory_not_link",EXTRACTED_BUNDLE_ROOT.is_dir() and not EXTRACTED_BUNDLE_ROOT.is_symlink())
        bundle=EXTRACTED_BUNDLE_ROOT
        report["bundle"]={"kind":"extracted_directory","path":str(bundle)}
    else:
        bundles=sorted(path for path in INPUT_ROOT.rglob(BUNDLE_NAME) if path.is_file())
        report["bundle_candidates"]=[str(path) for path in bundles]
        require("exactly_one_attached_original_zip",len(bundles)==1,
                f"No extracted setup directory at {EXTRACTED_BUNDLE_ROOT}; expected one original {BUNDLE_NAME} under {INPUT_ROOT}, found {len(bundles)}: {bundles}.")
        bundle=bundles[0]
        report["bundle"]={"kind":"zip","path":str(bundle),"sha256":file_hash(bundle)}
        require("zip_matches_reviewed_local_sha256",report["bundle"]["sha256"]==EXPECTED_BUNDLE_SHA256)
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 2. Verified setup copy/extraction and every payload checksum

Accept the extracted folder or original ZIP. Only the seven named members are allowed. Reject unexpected files, links, escaping paths, duplicate archive paths and destination conflicts. Verify the pinned manifest before parsing it, then verify its six payload checksums before copying anything. Preflight the entire destination; retain identical existing files and verify destination hashes before importing code. No recursive deletion or image copying is used.

In [ ]:
try:
    def safe_extract_bundle(bundle,destination):
        if destination.is_symlink():
            raise ValueError("Setup directory must not be a symlink")
        destination=destination.resolve()
        if not destination.is_relative_to(WORKING_ROOT.resolve()) or destination==WORKING_ROOT.resolve():
            raise ValueError("Setup directory must stay inside the dedicated working location")
        if bundle.is_symlink() or getattr(bundle,"is_junction",lambda:False)():
            raise ValueError("Bundle root must not be a link")
        if bundle.is_dir():
            bundle_root=bundle.resolve()
            allowed_directories={str(parent) for name in EXPECTED_MEMBERS for parent in PurePosixPath(name).parents if str(parent)!="."}
            source_files={}
            total_bytes=0
            for path in bundle.rglob("*"):
                relative=path.relative_to(bundle).as_posix()
                if path.is_symlink() or getattr(path,"is_junction",lambda:False)():
                    raise ValueError(f"Link in extracted bundle: {relative}")
                if not path.resolve().is_relative_to(bundle_root):
                    raise ValueError(f"Path escapes extracted bundle: {relative}")
                mode=path.lstat().st_mode
                if stat.S_ISDIR(mode):
                    if relative not in allowed_directories:
                        raise ValueError(f"Unexpected bundle directory: {relative}")
                elif stat.S_ISREG(mode):
                    if relative not in EXPECTED_MEMBERS:
                        raise ValueError(f"Unexpected bundle file: {relative}")
                    source_files[relative]=path
                    total_bytes+=path.stat().st_size
                else:
                    raise ValueError(f"Non-regular bundle member: {relative}")
            if set(source_files)!=EXPECTED_MEMBERS or total_bytes>100_000_000:
                raise ValueError("Missing members or unexpectedly large extracted bundle")
            contents={name:path.read_bytes() for name,path in source_files.items()}
        else:
            with zipfile.ZipFile(bundle) as archive:
                infos=archive.infolist()
                names=[item.filename for item in infos]
                if len(names)!=len(set(name.casefold() for name in names)) or set(names)!=EXPECTED_MEMBERS:
                    raise ValueError("Unexpected, missing or duplicate ZIP members")
                if sum(item.file_size for item in infos)>100_000_000:
                    raise ValueError("Unexpectedly large setup archive")
                for item in infos:
                    path=PurePosixPath(item.filename)
                    mode=item.external_attr>>16
                    if (path.is_absolute() or PureWindowsPath(item.filename).drive
                        or ".." in path.parts or "\\" in item.filename or ":" in item.filename
                        or item.is_dir() or stat.S_IFMT(mode) not in (0,stat.S_IFREG)):
                        raise ValueError(f"Unsafe archive member: {item.filename}")
                contents={item.filename:archive.read(item) for item in infos}
        if hashlib.sha256(contents["transfer_manifest.json"]).hexdigest()!=EXPECTED_TRANSFER_MANIFEST_SHA256:
            raise ValueError("Pinned transfer_manifest.json SHA256 mismatch")
        transfer=json.loads(contents["transfer_manifest.json"])
        expected=transfer["sha256"]
        if set(expected)!=EXPECTED_MEMBERS-{"transfer_manifest.json"}:
            raise ValueError("Transfer manifest has an unexpected checksum inventory")
        for name,digest in expected.items():
            if hashlib.sha256(contents[name]).hexdigest()!=digest:
                raise ValueError(f"Transferred checksum mismatch: {name}")
        if destination.exists():
            extras={path.relative_to(destination).as_posix() for path in destination.rglob("*") if path.is_file() or path.is_symlink()}-EXPECTED_MEMBERS
            if extras:
                raise ValueError(f"Unverified files in dedicated setup directory: {sorted(extras)}")
        plan=[]
        for name,data in contents.items():
            target=destination/name
            if not target.resolve().is_relative_to(destination):
                raise ValueError(f"Extraction leaves setup directory: {name}")
            for parent in (target,*target.parents):
                if parent==destination:
                    break
                if parent.is_symlink():
                    raise ValueError(f"Existing symlink in setup path: {parent}")
                if parent!=target and parent.exists() and not parent.is_dir():
                    raise ValueError(f"Non-directory parent: {parent}")
            if target.exists() and (not target.is_file() or target.read_bytes()!=data):
                raise FileExistsError(f"Conflicting existing file: {target}")
            plan.append((target,data))
        for target,data in plan:
            if not target.exists():
                target.parent.mkdir(parents=True,exist_ok=True)
                with target.open("xb") as stream:
                    stream.write(data)
        # The manifest pin covers the manifest itself for both input routes.
        actual={name:file_hash(destination/name) for name in EXPECTED_MEMBERS}
        for name,digest in expected.items():
            if actual[name]!=digest:
                raise ValueError(f"Extracted checksum mismatch: {name}")
        if actual["transfer_manifest.json"]!=hashlib.sha256(contents["transfer_manifest.json"]).hexdigest():
            raise ValueError("Extracted transfer manifest mismatch")
        return transfer,actual

    report["stage"]="safe extraction and artifact verification"
    transfer,artifact_hashes=safe_extract_bundle(bundle,SETUP_ROOT)
    report["artifact_sha256"]=artifact_hashes
    report["bundle"]["transfer_manifest_sha256"]=artifact_hashes["transfer_manifest.json"]
    require("pinned_transfer_manifest_verified",artifact_hashes["transfer_manifest.json"]==EXPECTED_TRANSFER_MANIFEST_SHA256)
    require("all_transferred_checksums_verified_before_import",all(artifact_hashes[name]==digest for name,digest in transfer["sha256"].items()))
    require("expected_bundle_contract",transfer["expected_counts"]=={"total":63660,"train":50927,"validation":12733}
            and transfer["trainable_parameters"]==94280 and transfer["seed"]==42)
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 3. Hosted versions and original code

Use the hosted packages as installed. Do not install the local requirements or upgrade packages. Report actual versions and their differences from the local CPU runtime. Source checksums have already passed. Disable new bytecode files in the dedicated setup directory, and reject a conflicting preloaded `src` package.

In [ ]:
try:
    report["stage"]="hosted runtime and verified imports"
    for distribution in ["torch","torchvision","numpy","pandas","Pillow"]:
        report["versions"][distribution]=version(distribution)
    report["source_versions"]=transfer["source_versions"]
    report["version_differences"]={name:{"source":value,"hosted":report["versions"].get(name)}
                                   for name,value in transfer["source_versions"].items() if report["versions"].get(name)!=value}
    save_report()
    # Avoid importing stale code from a different input or previous setup directory.
    for name,relative in [("src.models","src/models.py"),("src.data_pipeline","src/data_pipeline.py"),("src","src/__init__.py")]:
        loaded=sys.modules.get(name)
        if loaded is not None:
            if Path(getattr(loaded,"__file__","")).resolve()!=(SETUP_ROOT/relative).resolve():
                raise RuntimeError("A different src package is already loaded. Restart the session.")
            del sys.modules[name]
    sys.dont_write_bytecode=True
    sys.path.insert(0,str(SETUP_ROOT))
    importlib.invalidate_caches()
    import numpy as np
    import pandas as pd
    import torch
    import torchvision
    from PIL import Image
    from src.data_pipeline import FacialFeatureDataset, reference_preprocess, seed_everything
    from src.models import ReferenceCNN
    report["cuda_build"]=torch.version.cuda
    print(json.dumps(report["versions"],indent=2))
    print("Version differences:",json.dumps(report["version_differences"],indent=2))
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 4. Discover training nesting and verify the saved split

Locate `train_labels.csv` uniquely. Preserve all original string IDs and exact classes. Find candidate image directories using the largest saved ID as an anchor, then require exactly one directory with the complete expected filename set. This discovers nesting without assuming `train/train`. Only directory entries are examined during discovery; no test image contents are opened. All image decoding later is restricted to verified training IDs.

In [ ]:
try:
    report["stage"]="labels, training directory and fixed split"
    split=pd.read_csv(SETUP_ROOT/"data/processed/split_manifest.csv",dtype=str,keep_default_na=False,na_filter=False)
    mapping=json.loads((SETUP_ROOT/"data/processed/class_mapping.json").read_text(encoding="utf-8"))
    label_paths=sorted(path for path in DATA_ROOT.rglob("train_labels.csv") if path.is_file())
    require("unique_training_labels_csv",len(label_paths)==1)
    labels=pd.read_csv(label_paths[0],dtype=str,keep_default_na=False,na_filter=False)
    require("exact_training_label_columns",list(labels.columns)==["image_id","class"])
    require("original_ids_preserved_unique",labels["image_id"].is_unique and split["image_id"].is_unique
            and labels["image_id"].str.fullmatch(r"[0-9]{5}").all() and split["image_id"].str.fullmatch(r"[0-9]{5}").all())
    require("every_original_id_class_matches",labels.set_index("image_id").sort_index().equals(split[["image_id","class"]].set_index("image_id").sort_index()))
    actual_counts={"total":len(split),**{name:int(split["split"].eq(name).sum()) for name in ["train","validation"]}}
    require("saved_split_counts_and_full_coverage",actual_counts==transfer["expected_counts"] and split["split"].isin(["train","validation"]).all())
    classes=sorted(labels["class"].unique().tolist())
    expected_mapping={"class_to_index":{label:index for index,label in enumerate(classes)},
                      "index_to_class":{str(index):label for index,label in enumerate(classes)}}
    require("saved_class_encoding_unchanged",len(classes)==8 and mapping==transfer["classes"]==expected_mapping)
    require("all_classes_in_both_partitions",all(set(split.loc[split["split"].eq(name),"class"])==set(classes) for name in ["train","validation"]))
    require("valid_saved_hashes",all(split[column].str.fullmatch(r"[0-9a-f]{64}").all() for column in ["byte_sha256","pixel_sha256"]))
    require("original_manifest_paths_retained",split["image_path"].equals("data/raw/train/train/"+split["image_id"]+".jpg"))
    require("saved_pixel_groups_and_labels",split["group_id"].equals(split["pixel_sha256"])
            and split["pixel_sha256"].nunique()==transfer["pixel_groups"]
            and split.groupby("pixel_sha256")["class"].nunique().eq(1).all()
            and split.groupby("pixel_sha256")["split"].nunique().eq(1).all())
    train_rows=split[split["split"].eq("train")]
    validation_rows=split[split["split"].eq("validation")]
    overlaps={column:len(set(train_rows[column])&set(validation_rows[column])) for column in ["image_id","byte_sha256","pixel_sha256"]}
    report["cross_partition_overlap_counts"]=overlaps
    require("zero_cross_partition_overlap",not any(overlaps.values()))
    expected_filenames={image_id+".jpg" for image_id in labels["image_id"]}
    anchor=max(labels["image_id"])+".jpg"
    candidates=sorted({path.parent.resolve() for path in DATA_ROOT.rglob(anchor) if path.is_file()})
    matching=[]
    for directory in candidates:
        filenames={path.name for path in directory.glob("*.jpg") if path.is_file()}
        if filenames==expected_filenames:
            matching.append(directory)
    report["training_directory_candidates"]=[str(path) for path in candidates]
    require("exactly_one_complete_training_image_directory",len(matching)==1)
    TRAIN_IMAGE_DIR=matching[0]
    require("every_training_path_exists",all((TRAIN_IMAGE_DIR/name).is_file() and (TRAIN_IMAGE_DIR/name).resolve().parent==TRAIN_IMAGE_DIR for name in expected_filenames))
    train_dataset=FacialFeatureDataset(SETUP_ROOT,"train",training_image_directory=TRAIN_IMAGE_DIR)
    validation_dataset=FacialFeatureDataset(SETUP_ROOT,"validation",training_image_directory=TRAIN_IMAGE_DIR)
    require("portable_dataset_lengths_match",len(train_dataset)==50927 and len(validation_dataset)==12733)
    report["training_directory"]=str(TRAIN_IMAGE_DIR)
    report["training_labels_path"]=str(label_paths[0])
    report["counts"]=actual_counts
    report["mapping"]=mapping
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 5. Strict fingerprints before enabling the GPU check

Recompute byte and reference-preprocessed **CPU float32** tensor hashes on the same 12 notebook03 training images. Use the recorded shape, dtype and canonical hashing method. Report all differences and actual runtime versions. A mismatch stops the GPU check; do not change transforms, labels or tolerances to hide it.

In [ ]:
try:
    report["stage"]="strict preprocessing fingerprints"
    fingerprint_spec=json.loads((SETUP_ROOT/"preprocessing_fingerprints.json").read_text(encoding="utf-8"))
    examples=fingerprint_spec["examples"]
    require("twelve_unique_training_fingerprints",len(examples)==12 and len({item["image_id"] for item in examples})==12)
    fingerprint_results=[]
    for expected in examples:
        row=train_rows.loc[train_rows["image_id"].eq(expected["image_id"])]
        require("fingerprint_training_membership_"+expected["image_id"],len(row)==1 and row.iloc[0]["class"]==expected["class"])
        image_path=TRAIN_IMAGE_DIR/(expected["image_id"]+".jpg")
        actual_byte_hash=file_hash(image_path)
        with Image.open(image_path) as image:
            image.load()
            tensor=reference_preprocess(image)
        header=b"ENGE707-reference-f32-v1\0"+struct.pack(">3Q",*tensor.shape)
        pixels=np.asarray(tensor.detach().cpu().contiguous().numpy(),dtype="<f4")
        actual_tensor_hash=hashlib.sha256(header+pixels.tobytes(order="C")).hexdigest()
        fingerprint_results.append({"image_id":expected["image_id"],"original_byte_sha256":actual_byte_hash,
            "expected_original_byte_sha256":expected["original_byte_sha256"],"tensor_sha256":actual_tensor_hash,
            "expected_tensor_sha256":expected["tensor_sha256"],"shape":list(tensor.shape),"dtype":str(tensor.dtype),
            "byte_match":actual_byte_hash==expected["original_byte_sha256"],
            "tensor_match":actual_tensor_hash==expected["tensor_sha256"],
            "contract_match":list(tensor.shape)==expected["shape"]==[3,224,224] and tensor.dtype==torch.float32 and expected["dtype"]=="float32" and tensor.device.type=="cpu"})
    report["fingerprints"]=fingerprint_results
    report["fingerprint_method"]=fingerprint_spec["method"]
    save_report()
    require("all_strict_fingerprints_match",all(item["byte_match"] and item["tensor_match"] and item["contract_match"] for item in fingerprint_results))
    report["gpu_check_enabled"]=True
    save_report()
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise


## 6. Fresh original model: one inference-only batch on cuda:0

Select cuda:0 only, even if multiple GPUs are visible. Instantiate the original model fresh with seed 42. Take four verified training-partition images and use reference preprocessing directly, with no augmentation. Run one evaluation forward pass under `torch.inference_mode()`; do not create an optimiser or run backward. No weights are saved.

In [ ]:
try:
    report["stage"]="cuda:0 inference verification"
    require("fingerprints_gate_gpu_check",report.get("gpu_check_enabled",False))
    report["cuda_available"]=torch.cuda.is_available()
    report["cuda_device_count"]=torch.cuda.device_count()
    save_report()
    require("cuda_available",report["cuda_available"] and report["cuda_device_count"]>=1)
    torch.cuda.set_device(0)
    device=torch.device("cuda:0")
    free_bytes,total_bytes=torch.cuda.mem_get_info(0)
    report["gpu"]={"selected_device":"cuda:0","name":torch.cuda.get_device_name(0),
                   "available_memory_bytes":free_bytes,"total_memory_bytes":total_bytes}
    seed_everything(42)
    model=ReferenceCNN().to(device)
    parameter_count=sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    report["trainable_parameters"]=parameter_count
    require("original_parameter_count",parameter_count==94280==transfer["trainable_parameters"])
    require("model_uses_cuda0_only",all(parameter.device==device for parameter in model.parameters()))
    sample_rows=train_rows.head(4)
    tensors=[]
    for image_id in sample_rows["image_id"]:
        with Image.open(TRAIN_IMAGE_DIR/(image_id+".jpg")) as image:
            image.load()
            tensors.append(reference_preprocess(image))
    batch=torch.stack(tensors).to(device)
    model.eval()
    with torch.inference_mode():
        logits=model(batch)
    report["gpu_check_executed"]=True
    report["inference"]={"image_ids":sample_rows["image_id"].tolist(),"batch_shape":list(batch.shape),
                         "logit_shape":list(logits.shape),"device":str(logits.device),"finite":bool(torch.isfinite(logits).all()),
                         "requires_grad":logits.requires_grad}
    require("finite_eight_logit_output",list(logits.shape)==[len(sample_rows),8] and report["inference"]["finite"])
    require("eval_and_inference_only",not model.training and not logits.requires_grad and logits.is_inference())
    require("transferred_artifacts_remain_unchanged",all(file_hash(SETUP_ROOT/name)==digest for name,digest in artifact_hashes.items()))
    report["status"]="passed"
    save_report()
    del model,batch,logits,tensors
    print("Saved actual hosted results:",REPORT_PATH)
    print("GPU preparation checks passed. No training was performed.")
except Exception as error:
    report["failures"].append({"stage": report["stage"], "error": f"{type(error).__name__}: {error}"})
    report["status"] = "failed"
    save_report()
    raise
